# 删除与重建向量索引

本 Notebook 演示 LakeSoul 向量索引的 drop / rebuild / GC 完整流程：

- 创建带索引的小表；
- `dry_run` 预览；
- `keep_declaration=True` 删除并重建；
- 默认删除声明后 `recreate_vector_index()` 重建；
- 分区级删除；
- 异步清理和 GC；
- SQL 入口（dry-run 示例）。

完整参数说明见 [官网 API 文档](../../../website/docs/engine/vector-search/api.md)。

原生SQL batch使用独立的index_read_lease（TTL120秒、30秒续租）。drop立即阻止新的batch使用该索引；已固定快照的batch可完成。purge物理清理等待grace及活跃读租约结束，所以wait_for_cleanup=True可能超过grace_seconds；需要立即返回时使用False。

数据放在本地 `/tmp/lakesoul-tutorial/docs_drop_demo`，不依赖对象存储。


In [1]:
import os
import shutil
from pathlib import Path

TABLE = "docs_drop_demo"
ROOT = Path("/tmp/lakesoul-tutorial") / TABLE
shutil.rmtree(ROOT, ignore_errors=True)
ROOT.parent.mkdir(parents=True, exist_ok=True)

from lakesoul import LakeSoulCatalog

catalog = LakeSoulCatalog.from_env(namespace="default")
print("catalog:", catalog)


catalog: <lakesoul.catalog.LakeSoulCatalog object at 0x7ff9d81f2980>


## 1. 建表、写入、自动建索引

`create_table(vector_index=[...])` 会把配置写入 `vector_index_columns`；
`write_arrow` 默认会自动构建索引。


In [2]:
import pyarrow as pa

schema = pa.schema([
    pa.field("id", pa.int64(), nullable=False),
    pa.field("title", pa.string()),
    pa.field("embedding", pa.list_(pa.float32(), 8), nullable=False),
])

table = catalog.create_table(
    TABLE,
    path=str(ROOT),
    schema=schema,
    primary_keys=["id"],
    vector_index=[{
        "column": "embedding",
        "dim": 8,
        "nlist": 4,
        "total_bits": 7,
        "metric": "L2",
    }],
)

rows = [
    [1.0, 0, 0, 0, 0, 0, 0, 0],
    [0.9, 0.1, 0, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0, 0, 0, 0],
    [0, 0, 1, 0, 0, 0, 0, 0],
    [0, 0, 0, 1, 0, 0, 0, 0],
    [0.5, 0.5, 0, 0, 0, 0, 0, 0],
]
batch = pa.record_batch(
    [
        pa.array([1, 2, 3, 4, 5, 6], type=pa.int64()),
        pa.array(["a", "b", "c", "d", "e", "f"]),
        pa.array(rows, type=pa.list_(pa.float32(), 8)),
    ],
    schema=schema,
)
table.write_arrow(batch)

print("vector_index_columns:", table.vector_index_columns)
print("index root exists:", (ROOT / "_vector_index").exists())


  Reservoir: 3 / 3 vectors (0 MB)
  Training k-means (3 clusters, 15 iterations)...
  Streaming rotation + quantisation...
  Build complete: 3 vectors, 3 clusters
  ── Phase timing ──
    rotate reservoir:    0.0s
    k-means (15 iter):   0.0s
    stream rotate:       0.0s
    stream GEMM:         0.0s
    stream quantise:     0.0s
    stream append:       0.0s
    flush_pending:       0.0s
  Reservoir: 2 / 2 vectors (0 MB)
  Training k-means (2 clusters, 15 iterations)...
  Streaming rotation + quantisation...
  Build complete: 2 vectors, 2 clusters
  ── Phase timing ──
    rotate reservoir:    0.0s
    k-means (15 iter):   0.0s
    stream rotate:       0.0s
    stream GEMM:         0.0s
    stream quantise:     0.0s
    stream append:       0.0s
    flush_pending:       0.0s
vector_index_columns: ('embedding',)
  Reservoir: 1 / 1 vectors (0 MB)
  Training k-means (1 clusters, 15 iterations)...
  Streaming rotation + quantisation...
  Build complete: 1 vectors, 1 clusters
  ── Phase t

## 2. 查询基线

查询向量 `[1, 0, ...]` 的 top-3 是 `[1, 2, 6]`。


In [3]:
q = [1.0, 0, 0, 0, 0, 0, 0, 0]
print(table.search(q, top_k=3))


pyarrow.Table
id: int64 not null
title: string
----
id: [[1,2,6]]
title: [["a","b","f"]]


[FastScan] Using AVX-512 optimized path


## 3. dry-run 预览

`dry_run=True` 只返回将删除的 columns / partitions / shard 数，不修改任何元数据或对象。


In [4]:
preview = table.drop_vector_index(column="embedding", dry_run=True)
print(preview)


{'table_name': 'docs_drop_demo', 'namespace': 'default', 'columns': ['embedding'], 'partitions': ['-5'], 'shards_dropped': 3, 'objects_deleted': 0, 'bytes_deleted': 0, 'declaration_removed': False, 'dry_run': True, 'epoch': 0, 'drop_id': None, 'cleanup': ''}


## 4. 删除但保留声明，再 rebuild

`keep_declaration=True`：

- 删除物理索引；
- 保留 `vector_index_columns`；
- 清理完成后释放 tombstone；
- 之后可以直接 `rebuild_vector_index()`。


In [5]:
result = table.drop_vector_index(column="embedding", keep_declaration=True)
print(result)
print("declaration:", table.vector_index_columns)


{'table_name': 'docs_drop_demo', 'namespace': 'default', 'columns': ['embedding'], 'partitions': ['-5'], 'shards_dropped': 3, 'objects_deleted': 6, 'bytes_deleted': 6108, 'declaration_removed': False, 'dry_run': False, 'epoch': 1, 'drop_id': 'bffdda6e-5b16-43a2-998c-9a214591c682', 'cleanup': 'completed'}
declaration: ('embedding',)


In [6]:
# 删除后查询回退普通扫描，结果仍然正确
print(table.search(q, top_k=3))


pyarrow.Table
id: int64 not null
title: string
----
id: [[1,2,6]]
title: [["a","b","f"]]


In [7]:
result = table.rebuild_vector_index(column="embedding")
print(result)
print("vector_index_columns:", table.vector_index_columns)


  Reservoir: 3 / 3 vectors (0 MB)
  Training k-means (3 clusters, 15 iterations)...
  Streaming rotation + quantisation...
  Build complete: 3 vectors, 3 clusters
  ── Phase timing ──
    rotate reservoir:    0.0s
    k-means (15 iter):   0.0s
    stream rotate:       0.0s
    stream GEMM:         0.0s
    stream quantise:     0.0s
    stream append:       0.0s
    flush_pending:       0.0s
  Reservoir: 2 / 2 vectors (0 MB)
  Training k-means (2 clusters, 15 iterations)...
  Streaming rotation + quantisation...
  Build complete: 2 vectors, 2 clusters
  ── Phase timing ──
    rotate reservoir:    0.0s
    k-means (15 iter):   0.0s
    stream rotate:       0.0s
    stream GEMM:         0.0s
    stream quantise:     0.0s
    stream append:       0.0s
    flush_pending:       0.0s
  Reservoir: 1 / 1 vectors (0 MB)
  Training k-means (1 clusters, 15 iterations)...
  Streaming rotation + quantisation...
  Build complete: 1 vectors, 1 clusters
  ── Phase timing ──
    rotate reservoir:    0.0

## 5. 默认删除声明，再用 recreate 重建

默认 `keep_declaration=False`：声明被移除，tombstone 保留，防止旧 writer 重建。
需要重建时用 `recreate_vector_index()`：重新声明 + 清除 tombstone + rebuild。


In [8]:
result = table.drop_vector_index(column="embedding")
print(result)
print("declaration after drop:", table.vector_index_columns)


{'table_name': 'docs_drop_demo', 'namespace': 'default', 'columns': ['embedding'], 'partitions': ['-5'], 'shards_dropped': 3, 'objects_deleted': 6, 'bytes_deleted': 6108, 'declaration_removed': True, 'dry_run': False, 'epoch': 2, 'drop_id': '0f94b99e-ad59-422c-bcf7-176a7d514483', 'cleanup': 'tombstone_retained'}
declaration after drop: ()


In [9]:
result = table.recreate_vector_index(
    column="embedding",
    dim=8,
    nlist=4,
    total_bits=7,
    metric="L2",
)
print(result)
print("declaration after recreate:", table.vector_index_columns)
print(table.search(q, top_k=3))


  Reservoir: 3 / 3 vectors (0 MB)
  Training k-means (3 clusters, 15 iterations)...
  Streaming rotation + quantisation...
  Build complete: 3 vectors, 3 clusters
  ── Phase timing ──
    rotate reservoir:    0.0s
    k-means (15 iter):   0.0s
    stream rotate:       0.0s
    stream GEMM:         0.0s
    stream quantise:     0.0s
    stream append:       0.0s
    flush_pending:       0.0s
  Reservoir: 2 / 2 vectors (0 MB)
  Training k-means (2 clusters, 15 iterations)...
  Streaming rotation + quantisation...
  Build complete: 2 vectors, 2 clusters
  ── Phase timing ──
    rotate reservoir:    0.0s
    k-means (15 iter):   0.0s
    stream rotate:       0.0s
    stream GEMM:         0.0s
    stream quantise:     0.0s
    stream append:       0.0s
    flush_pending:       0.0s
{'status': 'ok', 'table_name': 'docs_drop_demo', 'columns': ['embedding'], 'results': [{'status': 'ok', 'table_name': 'docs_drop_demo', 'partitions_total': 1, 'partitions_processed': 1, 'details': [{'status': 'ok

## 6. 分区级删除

本演示表没有分区；有分区时使用：

```python
table.drop_vector_index(
    column="embedding",
    partition_desc="dt=2024-01-01",
    keep_declaration=True,
)
```

分区级删除必须保留声明。


## 7. 异步清理与 GC

`wait_for_cleanup=False` 时：

- 控制面立即删除；
- 返回 `cleanup="deferred"` 和 `drop_id`；
- 后台任务在 grace 后删除对象；
- 崩溃/失败后可手动 `gc_dropped_vector_index_objects()`。


In [10]:
import time

result = table.drop_vector_index(
    column="embedding",
    keep_declaration=True,
    grace_seconds=0.2,
    wait_for_cleanup=False,
)
print(result)
time.sleep(1.0)

print("gc:", table.gc_dropped_vector_index_objects())
print("declaration:", table.vector_index_columns)


{'table_name': 'docs_drop_demo', 'namespace': 'default', 'columns': ['embedding'], 'partitions': ['-5'], 'shards_dropped': 3, 'objects_deleted': 0, 'bytes_deleted': 0, 'declaration_removed': False, 'dry_run': False, 'epoch': 4, 'drop_id': '464b1650-8221-4a17-8a10-09c64a6c32a0', 'cleanup': 'deferred'}
gc: {'table_name': 'docs_drop_demo', 'namespace': 'default', 'tombstones': 0, 'objects_deleted': 0, 'bytes_deleted': 0}
declaration: ('embedding',)


## 8. SQL 入口（dry-run）

SQL 标量函数 `lakesoul_drop_vector_index` 的破坏性调用需要
`LAKESOUL_ALLOW_INDEX_DDL=1`；这里只演示 dry-run。

```sql
SELECT lakesoul_drop_vector_index(
    'docs_drop_demo', 'default', 'embedding',
    false, true, 0.0, true, true
);
```


In [11]:
session = catalog.session()
sql = """
SELECT lakesoul_drop_vector_index(
    'docs_drop_demo', 'default', 'embedding',
    false, true, 0.0, true, true
) AS report
"""
print(session.sql(sql).column("report").to_pylist()[0])


{"table_name":"docs_drop_demo","namespace":"default","columns":["embedding"],"partitions":[],"shards_dropped":0,"objects_deleted":0,"bytes_deleted":0,"declaration_removed":false,"dry_run":true,"epoch":4,"drop_id":null,"cleanup":""}


## 9. 清理

`drop_table` 会级联清理索引控制面和 tombstone；本地对象目录手动删除。


In [12]:
catalog.drop_table(TABLE, if_exists=True)
shutil.rmtree(ROOT, ignore_errors=True)
print("cleaned", ROOT)


cleaned /tmp/lakesoul-tutorial/docs_drop_demo
